# IOAI — 2025 Selection Camp Toxic Comments (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-toxic.zip', '/tmp/tox.zip')
    with zipfile.ZipFile('/tmp/tox.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Toxic Comments — 모범답안 (단어+글자 TF-IDF + 라벨별 임계값 튜닝)
원대회 reference 아이디어: 라벨별 분류기 + **정밀도-재현율 곡선으로 임계값 최적화**가 핵심(F1↑).
여기선 **단어 TF-IDF(1,2) + 글자 TF-IDF(char_wb 2-4)** 결합 + 라벨별 LogisticRegression, 그리고 held-out 보조분할로 **라벨별 최적 임계값**을 찾습니다. 채점 = held-out **mean_f1**(높을수록 좋음).

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
LABELS = ['toxic','severe_toxic','obscene','insult']
df = pd.read_csv('data/train.csv'); df['comment_text']=df['comment_text'].fillna('')
is_val = df['id'].apply(lambda s:int(str(s),16)%5==0)
tr, va = df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)
print('train',len(tr),'val',len(va))

In [ ]:
import numpy as np
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split

wvec = TfidfVectorizer(max_features=40000, ngram_range=(1,2), sublinear_tf=True)
cvec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=40000)
def feat(fit, texts):
    if fit: return hstack([wvec.fit_transform(texts), cvec.fit_transform(texts)])
    return hstack([wvec.transform(texts), cvec.transform(texts)])
# train 을 다시 fit/thr 로 나눠 임계값을 누수없이 튜닝
tr_f, tr_t = train_test_split(tr, test_size=0.2, random_state=0)
Xf = feat(True, tr_f['comment_text']); Xt = feat(False, tr_t['comment_text']); Xva = feat(False, va['comment_text'])
f1s, preds = [], {}
for c in LABELS:
    clf = LogisticRegression(max_iter=400, C=4, class_weight='balanced').fit(Xf, tr_f[c])
    p_t = clf.predict_proba(Xt)[:,1]
    pr, rc, th = precision_recall_curve(tr_t[c], p_t)
    f1c = (2*pr*rc/(pr+rc+1e-9))[:-1]; best_th = th[int(np.argmax(f1c))] if len(th) else 0.5
    p_va = (clf.predict_proba(Xva)[:,1] >= best_th).astype(int)
    preds[c] = p_va; f1s.append(f1_score(va[c], p_va, zero_division=0))
print('mean F1:', round(np.mean(f1s),4))
out = pd.DataFrame({'id': va['id']});
for c in LABELS: out[c] = preds[c]
out.to_csv('submission.csv', index=False); print('wrote submission.csv', len(out))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)